Directory three of the project:

```
├── code
├── data
├── jmxs
├── model
├── pcaps
└── results
```

The working dir of the script is the code/ directory

The code will be used to extract data to instrument the model. The data has been obtained from the system executing the application. The perform N readings in a observation time interval T. 

    1. read data from the data dir/ - 
    
    ```
    ├── 10-38
    │   ├── containers_pre.json
    │   ├── containers_post.json
    │   ├── energy.csv
    │   ├── system_pre.json
    │   ├── system_post.json
    │   └── requests.jtl
    ├── 11-38
    │   ├── containers_pre.json
    │   ├── containers_post.json
    │   ├── energy.csv
    │   ├── system_pre.json
    │   ├── system_post.json
    │   └── requests.jtl
    ```
       
    The name of the directory contains two numbers. The former describes the ith reading while the second one the population of customers operating while reading performance and energy values.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from itertools import chain
import csv, json, glob, os

In [2]:
IPSPATH = "experiment_configuration_data/ips.csv"
CONVERSATIONSPATH = "experiment_configuration_data/conversations.csv"
ips = pd.read_csv(IPSPATH)
conversations = pd.read_csv(CONVERSATIONSPATH)

In [3]:
for i, x in ips.iterrows():
    conversations = conversations.replace(x['ip'], x['container'])

In [4]:
conversations = pd.read_csv("experiment_configuration_data/conversations_names.csv")

In [5]:
conversations

,source,destination
0,145.108.225.16,145.108.225.7
1,145.108.225.16,ui
2,ui,auth
3,auth,verification-code
4,ui,verification-code
5,ui,travel
6,travel,ticketinfo
7,ticketinfo,basic
8,basic,station
9,travel,route


In [6]:
conversations[(conversations == 'travel').any(axis=1)]

,source,destination
5,ui,travel
6,travel,ticketinfo
9,travel,route
13,travel,order
14,travel,seat
16,travel,train
22,food,travel
29,preserve,travel


In [7]:
DATA = f"../model/results"
DIRS = list(map(lambda x: f"{DATA}/{x}", os.listdir(DATA)))
CUST = 75

In [8]:
DIRS = list(filter(lambda x: x.find(f"-{CUST}") != -1, DIRS))

In [9]:
DIRS

['../model/results/10-75',
 '../model/results/9-75',
 '../model/results/5-75',
 '../model/results/2-75',
 '../model/results/3-75',
 '../model/results/4-75',
 '../model/results/7-75',
 '../model/results/1-75',
 '../model/results/8-75',
 '../model/results/6-75']

In [10]:
def get_energy_stats(trials):
    time   = np.array([len(x['time']) for x in trials])
    energy = np.array(
        [np.trapz(x['power'], x['time']) for x in trials]
    )
    e = energy/time
    
    print(
            f"# Energy Per Visit(Joule/Visit):\n",
            f"## Mean:\t\t\t{energy.mean()}", 
            f"## Min-Max:\t\t\t[{energy.min()}, {energy.max()}]",
            f"## Var:\t\t\t\t{energy.var()}", 
            f"## Std:\t\t\t\t{energy.std()}", 
            '\n'
            f"# Average Response Time:\t{time.mean()}",
            f"# e (Joule/s):\t\t\t{e.mean()}, [{e.min()}, {e.max()}]",
            sep='\n'
         )

In [11]:
ENERGYFILES = [f"{x}/energy.csv" for x in DIRS]

In [12]:
energy_values = [
    pd.read_csv(x, names = ["time", "power"]) for x in ENERGYFILES
]

In [13]:
get_energy_stats(energy_values)

# Energy Per Visit(Joule/Visit):

## Mean:			746.967985
## Min-Max:			[723.2476499999998, 787.6673499999999]
## Var:				548.6473727090258
## Std:				23.423222935988672

# Average Response Time:	13.5
# e (Joule/s):			55.35127145604396, [53.17049285714286, 56.26195357142857]


In [14]:
from helpers.stats import SystemStats
from helpers.stats import ContainerStats

In [15]:
def calculate_containers_utilization(DIRS):
    rows = []
    for x in DIRS:
        f1, f2 = glob.glob(f"{x}/containers_*.json")
        data = ContainerStats(f2, f1).data
        rows.append(
            {k: [v['cpu'], v['disk'], v['io']] for k,v in data.items()}
        )

    return pd.DataFrame.from_records(rows)

def calculate_system_utilization(DIRS):
    rows = []
    for x in DIRS:
        f1, f2 = glob.glob(f"{x}/system_*.json")
        data = SystemStats(f2, f1).data[0]
        rows.append(
            {'cpu': data['cpu0'], 'disk': data['disk'], 'io': data['io'], 'duration': data['duration']}
        )

    return pd.DataFrame.from_records(rows)

In [16]:
containers_stats = calculate_containers_utilization(DIRS)
system_stats     = calculate_system_utilization(DIRS)
mean_duration    = system_stats['duration'].mean()
mean_disk        = system_stats['disk'].mean()
mean_io          = system_stats['io'].mean()
sys_mean_cpu     = system_stats['cpu'].mean()
arrival_rate     = CUST/mean_duration

In [17]:
print(
    f"mean_cpu: {sys_mean_cpu}",
    f"mean_disk: {mean_disk}",
    f"mean_duration: {mean_duration}",
    f"mean_io: {mean_io}",
    f"arrival_rate: {arrival_rate}"
)

mean_cpu: 46.7996 mean_disk: 16.0377 mean_duration: 14.224926829338074 mean_io: 570.5 arrival_rate: 5.272434853254705


In [18]:
#containers_stats.drop('docker', axis=1).head(1).values.sum()

In [23]:
tot = 0
for k in containers_stats.keys():
    cpu_util  = np.array([v[0] for v in containers_stats[k]])
    disk_util = np.array([v[1] for v in containers_stats[k]])
    io_util   = np.array([v[2] for v in containers_stats[k]])
    mean_cpu  = cpu_util.mean()
    mean_io   = io_util.mean()
    mean_disk = disk_util.mean()
    
    tot += mean_cpu

    print(
        f"{k:<50} {mean_cpu:<20} {((mean_cpu/100)/arrival_rate):<25}" 
    )
   
delta_util = (sys_mean_cpu - tot) / 100
print(f"total application utilization: {tot:>34} {delta_util} {delta_util/arrival_rate}")

baseline-ts-consign-price-service-1                0.10676923155236939  0.0002025046008609478    
baseline-ts-price-service-1                        0.37798032093966805  0.0007168989877728666    
baseline-ts-contacts-service-1                     0.6912866348222465   0.0013111335731261075    
baseline-ts-order-service-1                        1.623839920607007    0.0030798672070923767    
baseline-ts-route-service-1                        1.5439266339204405   0.002928299119651266     
baseline-ts-travel-service-1                       3.275018780153495    0.0062115870016522       
baseline-ts-user-service-1                         0.17224972971575278  0.0003266986402106458    
baseline-ts-config-service-1                       0.4387610061557397   0.0008321790944176199    
baseline-ts-ticketinfo-service-1                   1.9061076914482988   0.0036152323252920753    
baseline-ts-order-other-service-1                  0.15016305863928048  0.0002848078028817823    
baseline-ts-station-